<a href="https://colab.research.google.com/github/vi-xoxo/BigData26_KelasB_2311533010_Irgi-Fatihul-Ihsan/blob/main/Praktikum1/BD_KELAS_B_PO1_2311533010_IRGI_FATIHUL_IHSAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## J-1. Verifikasi Lingkungan Kerja

**Tujuan:** mengetahui persis spesifikasi mesin yang dipakai. Semua kesimpulan performa pada praktikum ini hanya berlaku untuk spesifikasi ini (CPMK-2).

In [ ]:
import sys, platform, multiprocessing

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("CPU core:", multiprocessing.cpu_count())

for nama in ["pandas", "numpy", "pyarrow", "matplotlib"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

In [ ]:
!free -h        # kapasitas dan sisa RAM
!df -h /content  # kapasitas dan sisa disk runtime
!java -version   # dibutuhkan PySpark pada langkah J-8

Spesifikasi di atas (versi Python/pandas, jumlah core, RAM, disk) adalah baseline pengukuran. Catat nilai-nilai ini di laporan karena setiap mesin Colab hampir pasti berbeda spesifikasinya.

## J-2. Menghubungkan Google Drive dan Menyiapkan Struktur Folder

Menyiapkan folder kerja sementara (`/content/data`, hilang saat runtime berakhir) dan folder permanen di Google Drive untuk menyimpan artefak.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")   # ikuti dialog izin yang muncul

import os
DIR_KERJA  = "/content/data"                              # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum1"   # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

 `DIR_KERJA` bersifat ephemeral (hilang saat runtime restart), sedangkan `DIR_SIMPAN` di Google Drive bersifat permanen. Semua artefak akhir (CSV, Parquet, PNG) wajib disimpan ke `DIR_SIMPAN` agar tidak hilang — ini prinsip *reproducibility* (Bagian F.5).

## J-3. Menyiapkan Alat Ukur Waktu dan Memori

Fungsi `ukur()` mencatat durasi eksekusi dan pertumbuhan RSS (*Resident Set Size* — memori fisik nyata yang dipakai proses) untuk setiap langkah, agar semua klaim performa pada laporan didukung angka, bukan asumsi.

In [ ]:
import time, os, psutil

proses = psutil.Process(os.getpid())
catatan = []   # menyimpan hasil pengukuran untuk dibandingkan nanti

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    """Menjalankan fungsi tanpa argumen, mencatat durasi dan pertumbuhan memori."""
    m0, t0 = rss_mb(), time.perf_counter()
    hasil  = fungsi()
    detik  = time.perf_counter() - t0
    delta  = rss_mb() - m0
    catatan.append({"langkah": label, "detik": round(detik, 2),
                     "delta_rss_mb": round(delta, 1)})
    print(f"[{label}] {detik:.2f} s | RSS {delta:+.1f} MB")
    return hasil

 RSS bisa naik-turun akibat *garbage collector*, sehingga dibaca sebagai indikasi besaran, bukan nilai eksak (lihat F). Setiap pemanggilan `ukur()` menambah satu baris ke `catatan`, yang nanti diekspor sebagai `pengukuran_kinerja.csv`.

## J-4. Mengunduh Dataset dan Inspeksi Awal

Mengunduh file Parquet NYC TLC Yellow Taxi Januari 2023 secara terprogram, lalu memeriksa metadata (ukuran file, jumlah baris, skema kolom) **tanpa memuat data ke memori**.

**Sumber data:** NYC Taxi & Limousine Commission — https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
**File:** `yellow_tripdata_2023-01.parquet` | **Tanggal unduh:** *(isi tanggal aktual)* | **Lisensi:** data publik NYC TLC

In [ ]:
import urllib.request

URL  = ("https://d37ci6vzurychx.cloudfront.net/trip-data/"
        "yellow_tripdata_2023-01.parquet")
PATH = os.path.join(DIR_KERJA, "yellow_tripdata_2023-01.parquet")

if not os.path.exists(PATH):
    ukur("unduh dataset", lambda: urllib.request.urlretrieve(URL, PATH))

print("Ukuran file:", round(os.path.getsize(PATH) / 1024**2, 1), "MB")

In [ ]:
import pyarrow.parquet as pq

meta = pq.ParquetFile(PATH).metadata
print("Jumlah baris :", f"{meta.num_rows:,}")
print("Jumlah kolom :", meta.num_columns)
print("Row group    :", meta.num_row_groups)
print("Nama kolom   :", pq.ParquetFile(PATH).schema.names)

Metadata (jumlah baris, kolom, skema) terbaca instan tanpa memuat satu baris pun ke memori — keunggulan pertama format kolumnar (Parquet), sesuai penjelasan L.1.

## J-4b. Jalur Cadangan Bila Unduhan Gagal

Jika jaringan laboratorium memblokir unduhan, sel di bawah membangkitkan dataset sintetis 5 juta baris secara lokal dengan skema yang sama. Sel ini otomatis dilewati jika `PATH` sudah berisi data hasil unduhan J-4.

> ⚠️ **Jika jalur ini terpakai**, seluruh interpretasi pola (mis. jam sibuk) pada langkah selanjutnya **tidak berlaku** karena distribusi data sintetis acak-seragam, bukan pola nyata.

In [ ]:
import numpy as np, pandas as pd

def bangkitkan_sintetis(n=5_000_000, seed=42):
    rng    = np.random.default_rng(seed)
    awal   = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31*24*60, n), unit="m")
    durasi = rng.integers(1, 90, n)
    return pd.DataFrame({
        "tpep_pickup_datetime":  pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count":       rng.integers(1, 5, n).astype("float64"),
        "trip_distance":         np.round(rng.gamma(2.0, 1.6, n), 2),
        "payment_type":          rng.integers(1, 5, n).astype("int64"),
        "fare_amount":           np.round(rng.gamma(4.0, 3.0, n), 2),
        "tip_amount":            np.round(rng.gamma(1.5, 1.2, n), 2),
    }).assign(total_amount=lambda d: (d.fare_amount + d.tip_amount + 3.0).round(2))

if not os.path.exists(PATH):
    bangkitkan_sintetis().to_parquet(PATH, index=False)
    print("Dataset sintetis dibuat di", PATH)
else:
    print("Dataset asli sudah tersedia — jalur cadangan dilewati.")

## J-5. Memuat Data dan Mengukur Biaya Memorinya

Membandingkan dua percobaan: memuat **semua kolom** vs memuat **hanya kolom yang dibutuhkan** (*column projection*), lalu menyempitkan tipe data (*downcasting*) untuk menghemat memori lebih lanjut (CPMK-3).

In [ ]:
import pandas as pd

KOLOM = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "passenger_count", "trip_distance", "payment_type",
         "fare_amount", "tip_amount", "total_amount"]

penuh = ukur("baca semua kolom", lambda: pd.read_parquet(PATH))
print("Dimensi:", penuh.shape)
print("Memori :", round(penuh.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

del penuh                       # bebaskan memori sebelum percobaan kedua
import gc; gc.collect()

df = ukur("baca kolom terpilih", lambda: pd.read_parquet(PATH, columns=KOLOM))
print("Dimensi:", df.shape)
print("Memori :", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
df.head()

In [ ]:
df.info(memory_usage="deep")
df.memory_usage(deep=True).sort_values(ascending=False) / 1024**2   # MB per kolom

In [ ]:
sebelum = df.memory_usage(deep=True).sum() / 1024**2

df["passenger_count"] = pd.to_numeric(df["passenger_count"], downcast="float")
df["payment_type"]    = df["payment_type"].astype("int8").astype("category")
for kol in ["trip_distance", "fare_amount", "tip_amount", "total_amount"]:
    df[kol] = pd.to_numeric(df[kol], downcast="float")

sesudah = df.memory_usage(deep=True).sum() / 1024**2
print(f"{sebelum:.1f} MB -> {sesudah:.1f} MB  (hemat {100*(1-sesudah/sebelum):.1f}%)")
df.dtypes

Memuat hanya kolom terpilih (*projection*) menurunkan pemakaian memori dibanding memuat semua kolom. Downcasting tipe numerik (`float64→float32`, `int→category`) menghemat memori lebih jauh, terutama pada kolom berkardinalitas rendah seperti `payment_type` (L.2). Trade-off: `downcast="float"` mengurangi presisi — tidak boleh dipakai untuk perhitungan tagihan eksak.

## J-6. Pemeriksaan Kualitas Data dan Pembersihan (Veracity)

Menghitung durasi perjalanan, memeriksa nilai kosong, mendeteksi anomali (jarak/durasi/tarif tidak masuk akal), lalu menyaring baris yang layak dianalisis. Ambang batas (*threshold*) di bawah adalah keputusan analitis, bukan kebenaran mutlak.

In [ ]:
df["durasi_menit"] = (df["tpep_dropoff_datetime"] -
                       df["tpep_pickup_datetime"]).dt.total_seconds() / 60

print(df[["trip_distance", "durasi_menit", "total_amount"]].describe())
print("\nNilai kosong per kolom:\n", df.isna().sum())

anomali = {
    "jarak <= 0"        : (df["trip_distance"] <= 0).sum(),
    "durasi <= 0"       : (df["durasi_menit"]  <= 0).sum(),
    "durasi > 180 menit": (df["durasi_menit"]  > 180).sum(),
    "total_amount <= 0" : (df["total_amount"]  <= 0).sum(),
}
for k, v in anomali.items():
    print(f"{k:22s}: {v:,} baris ({100*v/len(df):.2f}%)")

In [ ]:
layak = (
    (df["trip_distance"] > 0) & (df["trip_distance"] < 100) &
    df["durasi_menit"].between(1, 180) &
    (df["total_amount"] > 0)
)
bersih = df.loc[layak].copy()
print(f"{len(df):,} baris -> {len(bersih):,} baris  (dibuang {len(df)-len(bersih):,})")

Ambang batas dipilih berdasarkan penalaran domain, misalnya durasi > 180 menit dianggap *error recording* karena tidak masuk akal untuk perjalanan taksi dalam kota. Baris anomali (jarak/durasi/tarif ≤ 0) mencerminkan dimensi **Veracity** pada 5V (Bagian F.1).

## J-7. Memproses File Lebih Besar dari Memori dengan Chunking

Mensimulasikan kondisi "file terlalu besar": menulis ulang data bersih sebagai CSV (jauh lebih besar dari Parquet), lalu memproses file itu *bagian demi bagian* (streaming aggregation) tanpa pernah memuatnya secara utuh — inti gagasan pemrosesan *out-of-core* (CPMK-3, L.3).

In [ ]:
CSV = os.path.join(DIR_KERJA, "trips.csv")
ukur("tulis CSV", lambda: bersih.to_csv(CSV, index=False))

print("Parquet:", round(os.path.getsize(PATH) / 1024**2, 1), "MB")
print("CSV    :", round(os.path.getsize(CSV)  / 1024**2, 1), "MB")

In [ ]:
def agregasi_bertahap(path, ukuran_chunk=500_000):
    jumlah_baris = 0
    total_nilai  = 0.0
    per_jam      = {}
    potongan = pd.read_csv(
        path,
        usecols=["tpep_pickup_datetime", "total_amount"],
        parse_dates=["tpep_pickup_datetime"],
        chunksize=ukuran_chunk)
    for chunk in potongan:
        jumlah_baris += len(chunk)
        total_nilai  += chunk["total_amount"].sum()
        for jam, sub in chunk.groupby(chunk["tpep_pickup_datetime"].dt.hour):
            n, nilai = per_jam.get(jam, (0, 0.0))
            per_jam[jam] = (n + len(sub), nilai + sub["total_amount"].sum())
    return jumlah_baris, total_nilai, per_jam

baris, total, per_jam_chunk = ukur("agregasi chunking", lambda: agregasi_bertahap(CSV))
print(f"{baris:,} baris | rata-rata total_amount = {total/baris:.2f}")

CSV jauh lebih besar dari Parquet karena berupa teks tanpa kompresi. Nilai `delta_rss_mb` pada hasil `ukur()` untuk langkah chunking tetap kecil meski jumlah baris besar, karena hanya satu potongan berada di memori pada satu waktu — bukti konsep *streaming aggregation* / *out-of-core*.

## J-8. Pendekatan Terdistribusi: PySpark Local Mode

Menjalankan agregasi yang sama menggunakan PySpark dalam mode `local[*]` (satu mesin, memakai semua core) untuk membandingkan paradigma *distributed processing* dengan pandas (CPMK-6, L.4).

> Diuji dengan PySpark 3.5.1 di atas OpenJDK 11/17. Jika `!java -version` pada J-1 menunjukkan Java 8, tetap gunakan PySpark 3.5.x — **jangan** memasang PySpark 4.x pada modul ini.

In [ ]:
!pip install -q pyspark==3.5.1
!java -version

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("BD-P01")
         .master("local[*]")                 # semua core mesin ini
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

In [ ]:
sdf = spark.read.parquet(PATH)
sdf.printSchema()
print("Jumlah baris:", ukur("spark count", lambda: sdf.count()))

sdf_bersih = (sdf
    .withColumn(
        "durasi_menit",
        (F.col("tpep_dropoff_datetime").cast("long")
         - F.col("tpep_pickup_datetime").cast("long")) / 60)
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    .filter((F.col("durasi_menit") >= 1) & (F.col("durasi_menit") <= 180))
    .filter(F.col("total_amount") > 0))

hasil_spark = (sdf_bersih
    .withColumn("jam", F.hour("tpep_pickup_datetime"))
    .groupBy("jam")
    .agg(F.count("*").alias("jumlah"),
         F.round(F.avg("trip_distance"), 2).alias("rata_jarak"),
         F.round(F.avg("total_amount"), 2).alias("rata_tarif"))
    .orderBy("jam"))

ukur("spark agregasi", lambda: hasil_spark.show(24, truncate=False))

In [ ]:
sdf_bersih.createOrReplaceTempView("trips")
spark.sql("""
    SELECT payment_type,
           COUNT(*)                    AS jumlah,
           ROUND(AVG(total_amount), 2) AS rata_tarif,
           ROUND(AVG(tip_amount), 2)   AS rata_tip
    FROM trips
    GROUP BY payment_type
    ORDER BY jumlah DESC
""").show()

hasil_spark.explain()      # rencana eksekusi — perhatikan Exchange / HashAggregate
spark.stop()                # bebaskan sumber daya sebelum lanjut

`sdf.count()` memakan waktu lebih lama dari dugaan karena Spark bersifat *lazy* — pekerjaan baru berjalan saat ada *action* (count/show/write). Untuk data sebesar ini, pandas kemungkinan besar lebih cepat karena Spark menambah overhead penjadwalan dan serialisasi — ini bukan kegagalan Spark, melainkan bukti bahwa alat harus dipilih sesuai skala data (F.2).

## J-9. Agregasi Akhir dengan Pandas dan Penyimpanan Artefak

Menghasilkan tabel agregasi per jam (jumlah perjalanan, rata-rata jarak/durasi/tarif, median tarif) dan menyimpannya sebagai CSV + Parquet — dua dari tiga artefak wajib yang dikumpulkan (CPMK-3, CPMK-5).

In [ ]:
bersih["jam"] = bersih["tpep_pickup_datetime"].dt.hour

agregasi = (bersih
    .groupby("jam")
    .agg(jumlah_perjalanan=("total_amount", "size"),
         rata_jarak=("trip_distance", "mean"),
         rata_durasi=("durasi_menit", "mean"),
         rata_tarif=("total_amount", "mean"),
         median_tarif=("total_amount", "median"))
    .round(2)
    .reset_index())

agregasi.to_csv(f"{DIR_SIMPAN}/agregasi_per_jam.csv", index=False)
agregasi.to_parquet(f"{DIR_SIMPAN}/agregasi_per_jam.parquet", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)

agregasi

Tiga file (`agregasi_per_jam.csv`, `agregasi_per_jam.parquet`, `pengukuran_kinerja.csv`) tersimpan permanen di Google Drive. Pastikan ketiganya benar-benar ada sebelum menutup runtime.

## J-10. Visualisasi Hasil

Satu diagram batang jumlah perjalanan per jam — satu grafik, satu pesan, tanpa menumpuk metrik berskala berbeda dalam satu sumbu.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(agregasi["jam"], agregasi["jumlah_perjalanan"])
ax.set_xlabel("Jam penjemputan (0-23)")
ax.set_ylabel("Jumlah perjalanan")
ax.set_title("Distribusi perjalanan per jam")
ax.set_xticks(range(0, 24))
fig.tight_layout()
fig.savefig(f"{DIR_SIMPAN}/distribusi_per_jam.png", dpi=150)
plt.show()

Grafik menunjukkan pola permintaan sepanjang hari — jam dengan batang tertinggi mengindikasikan jam sibuk. Nilai aktual bergantung pada data eksekusi masing-masing mahasiswa.

## N. Analisis Hasil

Menyusun tabel perbandingan dari `pd.DataFrame(catatan)` (hasil `ukur()` di J-1–J-9), lalu menjawab enam pertanyaan analisis dengan kode + angka aktual (aturan: satu klaim, satu angka, satu alasan).

In [ ]:
tabel_kinerja = pd.DataFrame(catatan)
tabel_kinerja

In [ ]:
# 1. Biaya membaca data: rasio waktu & memori (semua kolom vs kolom terpilih)
t_semua   = tabel_kinerja.loc[tabel_kinerja.langkah == "baca semua kolom", "detik"].values[0]
t_terpilih = tabel_kinerja.loc[tabel_kinerja.langkah == "baca kolom terpilih", "detik"].values[0]
rasio_waktu  = t_semua / t_terpilih
rasio_kolom  = pq.ParquetFile(PATH).metadata.num_columns / len(KOLOM)

print(f"Waktu baca semua kolom   : {t_semua:.2f} s")
print(f"Waktu baca kolom terpilih: {t_terpilih:.2f} s")
print(f"Rasio waktu   : {rasio_waktu:.2f}x")
print(f"Rasio jumlah kolom (total/terpilih): {rasio_kolom:.2f}x")
print("-> Rasio waktu tidak selalu identik dengan rasio kolom karena Parquet melakukan"
      " column pruning: waktu baca dipengaruhi ukuran blok kolom, bukan sekadar jumlahnya (L.1).")

In [ ]:
# 2. Efek optimasi tipe data
print(f"Memori sebelum optimasi: {sebelum:.1f} MB")
print(f"Memori sesudah optimasi: {sesudah:.1f} MB")
print(f"Persentase hemat        : {100*(1-sesudah/sebelum):.1f}%")

kontribusi = df.memory_usage(deep=True).sort_values(ascending=False) / 1024**2
print("\nKontribusi memori terbesar (setelah optimasi):")
print(kontribusi.head(3))
print("-> Risiko: downcast float64->float32 mengurangi presisi angka uang;"
      " boleh untuk agregat, tidak untuk perhitungan tagihan eksak (L.2).")

In [ ]:
# 3. Format penyimpanan: rasio ukuran CSV vs Parquet
ukuran_parquet = os.path.getsize(PATH) / 1024**2
ukuran_csv     = os.path.getsize(CSV)  / 1024**2
print(f"Parquet: {ukuran_parquet:.1f} MB | CSV: {ukuran_csv:.1f} MB")
print(f"Rasio CSV/Parquet: {ukuran_csv/ukuran_parquet:.2f}x lebih besar")
print("-> Dua penyebab: (1) CSV menyimpan teks tanpa kompresi kolomnar;"
      " (2) Parquet mengompresi tiap kolom dengan encoding sesuai tipe datanya (F.4).")

In [ ]:
# 4. Chunking: delta_rss_mb pemuatan penuh vs agregasi chunking
delta_penuh    = tabel_kinerja.loc[tabel_kinerja.langkah == "baca semua kolom", "delta_rss_mb"].values[0]
delta_chunking = tabel_kinerja.loc[tabel_kinerja.langkah == "agregasi chunking", "delta_rss_mb"].values[0]
print(f"delta_rss_mb (baca semua kolom) : {delta_penuh:+.1f} MB")
print(f"delta_rss_mb (agregasi chunking): {delta_chunking:+.1f} MB")
print("-> Pengorbanan: agregasi harus bersifat associative (sum/count/min/max);"
      " median/nilai unik eksak tidak bisa dihitung langsung per potongan (L.3).")

In [ ]:
# 5. pandas vs Spark: waktu agregasi
t_pandas = tabel_kinerja.loc[tabel_kinerja.langkah == "agregasi chunking", "detik"].values[0]
t_spark  = tabel_kinerja.loc[tabel_kinerja.langkah == "spark agregasi", "detik"].values[0] \
           if "spark agregasi" in tabel_kinerja.langkah.values else None
print(f"Waktu pandas (chunking): {t_pandas:.2f} s")
print(f"Waktu spark  (agregasi): {t_spark:.2f} s" if t_spark else "Spark tidak dijalankan pada sesi ini")
print("-> Pada data skala puluhan-ratusan MB, pandas umumnya lebih cepat karena Spark"
      " menanggung overhead penjadwalan & serialisasi (L.4). Titik balik diperkirakan"
      " terjadi saat data tidak lagi muat di satu mesin (>puluhan GB / >RAM tersedia),"
      " karena di situ paralelisme & distribusi mulai mengungguli overhead-nya.")

In [ ]:
# 6. Kualitas data: persentase baris dibuang & potensi bias
persen_dibuang = 100 * (len(df) - len(bersih)) / len(df)
print(f"Baris dibuang: {len(df)-len(bersih):,} dari {len(df):,} ({persen_dibuang:.2f}%)")
print("-> Potensi bias sistematis: jika baris trip_distance ekstrem/nol ikut terbuang"
      " secara tidak proporsional pada jam/area tertentu, kesimpulan pola per jam bisa"
      " condong pada perjalanan 'normal' saja dan meremehkan kasus tepi (edge case).")

## O. Studi Kasus — Penjadwalan Armada oleh Operator Taksi

**Kasus:** Operator dengan 1.200 armada perlu menyusun jadwal shift untuk kuartal berikutnya. Tiga pertanyaan manajemen: (1) jam permintaan tertinggi/terendah, (2) perbedaan pola hari kerja vs akhir pekan, (3) apakah jam sibuk lebih pendek namun lebih menguntungkan per menit. Dikerjakan dalam satu mesin, tanpa klaster, menggunakan `bersih` dari J-6.

In [ ]:
bersih["hari"]        = bersih["tpep_pickup_datetime"].dt.day_name()
bersih["akhir_pekan"]  = bersih["tpep_pickup_datetime"].dt.dayofweek >= 5
bersih["tarif_per_menit"] = (bersih["total_amount"] / bersih["durasi_menit"]).round(3)

ringkas = (bersih
    .groupby(["akhir_pekan", "jam"])
    .agg(jumlah=("total_amount", "size"),
         rata_durasi=("durasi_menit", "mean"),
         rata_tarif_per_menit=("tarif_per_menit", "mean"))
    .round(2))
ringkas.head(12)

In [ ]:
# (1) Jam permintaan tertinggi & terendah (gabungan semua hari)
per_jam_total = bersih.groupby("jam")["total_amount"].size()
jam_tertinggi = per_jam_total.idxmax()
jam_terendah  = per_jam_total.idxmin()
print(f"Jam permintaan tertinggi: {jam_tertinggi}:00 ({per_jam_total.max():,} perjalanan)")
print(f"Jam permintaan terendah : {jam_terendah}:00 ({per_jam_total.min():,} perjalanan)")

In [ ]:
# (2) Perbedaan pola hari kerja vs akhir pekan
pola = (bersih.groupby(["akhir_pekan", "jam"])["total_amount"].size()
        .unstack(level=0).rename(columns={False: "hari_kerja", True: "akhir_pekan"}))
jam_puncak_kerja  = pola["hari_kerja"].idxmax()
jam_puncak_akhir  = pola["akhir_pekan"].idxmax()
print(f"Jam puncak hari kerja  : {jam_puncak_kerja}:00")
print(f"Jam puncak akhir pekan : {jam_puncak_akhir}:00")
print("-> Pola berbeda jika kedua jam puncak tidak sama.")

In [ ]:
# (3) Apakah jam sibuk lebih pendek namun lebih menguntungkan per menit?
jam_sibuk = per_jam_total.idxmax()
durasi_sibuk = bersih.loc[bersih.jam == jam_sibuk, "durasi_menit"].mean()
durasi_rata2 = bersih["durasi_menit"].mean()
tarif_sibuk  = bersih.loc[bersih.jam == jam_sibuk, "tarif_per_menit"].mean()
tarif_rata2  = bersih["tarif_per_menit"].mean()

print(f"Durasi rata-rata jam sibuk ({jam_sibuk}:00): {durasi_sibuk:.2f} menit  | rata-rata umum: {durasi_rata2:.2f} menit")
print(f"Tarif per menit jam sibuk : {tarif_sibuk:.3f}  | rata-rata umum: {tarif_rata2:.3f}")

In [ ]:
# Tiga rekomendasi operasional (satu kalimat, satu angka pendukung, satu batasan)
print(f"1. Tambah armada pada jam {jam_tertinggi}:00 karena volume permintaan tertinggi "
      f"({per_jam_total.max():,} perjalanan); catatan: data hanya satu bulan musim dingin, "
      f"pola musiman belum tentu berlaku.")
print(f"2. Kurangi armada pada jam {jam_terendah}:00 karena permintaan terendah "
      f"({per_jam_total.min():,} perjalanan); catatan: hari libur/cuaca ekstrem tidak "
      f"dipisahkan dalam analisis ini.")
print(f"3. Prioritaskan pengemudi berpengalaman pada jam sibuk karena tarif per menit "
      f"{'lebih tinggi' if tarif_sibuk > tarif_rata2 else 'tidak lebih tinggi'} "
      f"({tarif_sibuk:.3f} vs rata-rata {tarif_rata2:.3f}); catatan: tarif per menit "
      f"belum memperhitungkan biaya bahan bakar/waktu tunggu pengemudi.")

## P. Latihan

Dikerjakan langsung di notebook yang sama setelah J-10 selesai.

### Latihan 1 — 5 nilai `trip_distance` terbesar

In [ ]:
top5_jarak = bersih.nlargest(5, "trip_distance")
top5_jarak

Jika nilai `trip_distance` jauh melebihi rentang wajar perjalanan dalam kota, kemungkinan itu tetap merupakan kesalahan pencatatan meski lolos ambang batas 100 mil pada J-6.

### Latihan 2 — Ulangi J-7 dengan `chunksize=100_000` dan `chunksize=1_000_000`

In [ ]:
hasil_ukuran_chunk = {}
for ukuran in [100_000, 1_000_000]:
    label = f"chunking size={ukuran}"
    _ = ukur(label, lambda u=ukuran: agregasi_bertahap(CSV, ukuran_chunk=u))
    hasil_ukuran_chunk[ukuran] = catatan[-1]

pd.DataFrame(catatan).tail(2)

`chunksize` kecil (100.000) menekan pemakaian memori tapi menambah overhead per-iterasi sehingga waktu total cenderung lebih lama; `chunksize` besar (1.000.000) sebaliknya — lebih cepat tapi memori per potongan lebih besar. Ukuran optimal bergantung pada batas RAM yang tersedia.

### Latihan 3 — Persentase tip per metode pembayaran

In [ ]:
bersih["persen_tip"] = np.where(
    bersih["fare_amount"] > 0,
    bersih["tip_amount"] / bersih["fare_amount"] * 100,
    np.nan)   # hindari pembagian dengan nol

rata_tip_per_metode = bersih.groupby("payment_type")["persen_tip"].mean().round(2)
rata_tip_per_metode

Metode pembayaran non-tunai (kartu kredit) cenderung memiliki persentase tip rata-rata lebih tinggi dibanding tunai, karena aplikasi pembayaran digital sering menampilkan saran nominal tip secara otomatis.

### Latihan 4 — Simpan Parquet dengan dua kompresi berbeda

In [ ]:
import time

hasil_kompresi = []
for komp in ["snappy", "gzip"]:
    path_out = f"{DIR_KERJA}/bersih_{komp}.parquet"
    t0 = time.perf_counter()
    bersih.to_parquet(path_out, index=False, compression=komp)
    durasi = time.perf_counter() - t0
    ukuran = os.path.getsize(path_out) / 1024**2
    hasil_kompresi.append({"kompresi": komp, "ukuran_MB": round(ukuran, 1), "waktu_tulis_s": round(durasi, 2)})

pd.DataFrame(hasil_kompresi)

`gzip` umumnya menghasilkan file lebih kecil daripada `snappy`, tetapi dengan waktu tulis (dan baca) yang lebih lama karena rasio kompresi lebih tinggi membutuhkan komputasi lebih besar — trade-off ruang penyimpanan vs kecepatan.

### Latihan 5 — Verifikasi konsistensi agregasi pandas vs Spark SQL

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("BD-P01-L5")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH)
sdf_bersih_l5 = (sdf
    .withColumn("durasi_menit",
                (F.col("tpep_dropoff_datetime").cast("long")
                 - F.col("tpep_pickup_datetime").cast("long")) / 60)
    .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
    .filter((F.col("durasi_menit") >= 1) & (F.col("durasi_menit") <= 180))
    .filter(F.col("total_amount") > 0)
    .withColumn("jam", F.hour("tpep_pickup_datetime")))

agregasi_spark = (sdf_bersih_l5.groupBy("jam")
    .agg(F.count("*").alias("jumlah_perjalanan"))
    .orderBy("jam")
    .toPandas())

pembanding = (agregasi[["jam", "jumlah_perjalanan"]]
              .merge(agregasi_spark, on="jam", suffixes=("_pandas", "_spark")))
pembanding["selisih"] = pembanding.jumlah_perjalanan_pandas - pembanding.jumlah_perjalanan_spark
print("Selisih maksimum:", pembanding["selisih"].abs().max())
spark.stop()
pembanding

Selisih maksimum 0 membuktikan agregasi pandas dan Spark SQL konsisten. Jika ada selisih, kemungkinan penyebabnya adalah perbedaan penanganan nilai batas (*inclusive/exclusive*) pada filter durasi antara kedua implementasi.

## Ringkasan Kesimpulan

1. **"Big" adalah hubungan antara data dan sumber daya** — harus diukur, bukan ditebak (F.1).
2. Sebagian besar masalah skala pada tahap awal dapat diselesaikan **tanpa klaster**: memilih kolom, tipe data, format penyimpanan, dan pemrosesan bertahap (chunking) sudah memberi penghematan besar.
3. **Spark cocok untuk skala tertentu** — pada data yang masih muat di satu mesin, pandas umumnya lebih cepat karena Spark menanggung overhead penjadwalan & serialisasi.

**Artefak yang dihasilkan di Google Drive (`DIR_SIMPAN`):**
- `agregasi_per_jam.csv` & `agregasi_per_jam.parquet`
- `pengukuran_kinerja.csv`
- `distribusi_per_jam.png`
